# Module `algorithms/genetic.py`

Algorithme genetique **a codage giant-tour** : un individu est une permutation des clients, pas directement une liste de sous-tournees. La fonction `_split` (Prins, 2004) decode cette permutation en solution VRP **optimale sous contrainte de l'ordre fige** par programmation dynamique en O(n^2).

Pipeline :

1. Population initiale : permutations aleatoires.
2. Boucle generationnelle : tournoi -> OX crossover -> mutation (swap ou inversion) -> Split -> `local_search` optionnelle (schema **memetique**).
3. Remplacement avec elitisme.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import genetic_algorithm
from cesipath.algorithms.genetic import _ox_crossover, _mutate, _split
from cesipath.solver_input import build_static_solver_input
import random


## 1. Instance de travail


In [2]:
cfg = GraphGenerationConfig(node_count=15, seed=4)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)
print(f"n = {instance.node_count}, capacite = {si.vehicle_capacity}")


n = 15, capacite = 40


## 2. Codage giant-tour + Split de Prins

Un *giant-tour* est une permutation des clients. Le Split decoupe cette permutation en sous-tournees admissibles de cout minimal par programmation dynamique :

- `v[j]` = cout minimum pour couvrir les `j` premiers clients de la permutation.
- `p[j]` = point de decoupe optimal precedent.

Le Split est *optimal sous l'ordre impose* : il ne re-ordonne pas les clients, il choisit uniquement ou inserer les retours au depot.


In [3]:
depot = si.depot
clients = [c for c, d in si.demands.items() if c != depot and d > 0]
rng = random.Random(1)
perm = clients[:]
rng.shuffle(perm)

routes, cost = _split(perm, depot, si.cost_matrix, si.demands, si.vehicle_capacity)
print("giant-tour :", perm)
print(f"cout Split : {cost:.2f}  en {len(routes)} sous-tournees")
for r in routes:
    print("  ", r)


giant-tour : [14, 11, 1, 13, 7, 6, 4, 9, 8, 12, 5, 2, 10, 3]
cout Split : 1687.68  en 8 sous-tournees
   [0, 14, 0]
   [0, 11, 1, 0]
   [0, 13, 7, 0]
   [0, 6, 4, 0]
   [0, 9, 8, 0]
   [0, 12, 5, 0]
   [0, 2, 0]
   [0, 10, 3, 0]


## 3. OX crossover (Order Crossover)

OX copie un segment du parent 1, puis complete dans l'ordre du parent 2 en sautant les clients deja pris. Il preserve un sous-ordre d'un parent et l'ordre relatif restant de l'autre.


In [4]:
rng = random.Random(42)
p1 = clients[:]
p2 = clients[:]
rng.shuffle(p1)
rng.shuffle(p2)
child = _ox_crossover(p1, p2, rng)
print("p1    :", p1)
print("p2    :", p2)
print("child :", child)
print("meme multiset ?", sorted(child) == sorted(clients))


p1    : [9, 13, 8, 7, 14, 12, 6, 3, 10, 4, 5, 1, 2, 11]
p2    : [3, 6, 8, 10, 7, 12, 5, 13, 9, 11, 4, 2, 14, 1]
child : [5, 13, 9, 11, 14, 12, 6, 3, 10, 4, 2, 1, 8, 7]
meme multiset ? True


## 4. Mutation : swap ou inversion

Avec probabilite 0.5, soit on echange deux genes, soit on inverse un segment. L'inversion preserve les adjacences internes sauf aux deux extremites et est donc moins destructive que le swap pur.


In [5]:
rng = random.Random(0)
for _ in range(3):
    muted = _mutate(clients[:], rng)
    print(muted)


[1, 2, 3, 4, 5, 6, 13, 12, 11, 10, 9, 8, 7, 14]
[1, 2, 3, 4, 5, 6, 7, 9, 8, 10, 11, 12, 13, 14]
[1, 2, 3, 4, 13, 6, 7, 8, 9, 10, 11, 12, 5, 14]


## 5. Boucle generationnelle complete


In [6]:
sol = genetic_algorithm(
    si,
    population_size=30,
    generations=50,
    tournament_k=3,
    crossover_rate=0.85,
    mutation_rate=0.2,
    elitism=2,
    memetic=True,
    seed=0,
)
print(f"cout = {sol.total_cost:.2f}  routes = {sol.route_count}")


cout = 1448.30  routes = 7


## 6. Effet du schema memetique

L'option `memetic=True` applique `local_search` a chaque enfant apres Split. C'est nettement plus couteux (chaque evaluation devient une descente complete) mais massivement plus robuste : chaque individu est deja un optimum local.


In [7]:
import time

for memetic in (False, True):
    t0 = time.perf_counter()
    sol = genetic_algorithm(si, population_size=30, generations=40, memetic=memetic, seed=0)
    dt = time.perf_counter() - t0
    print(f"memetic={memetic!s:<5} -> cout={sol.total_cost:.2f}  temps={dt*1000:.1f} ms")


memetic=False -> cout=1481.98  temps=15.8 ms


memetic=True  -> cout=1448.30  temps=489.2 ms


## 7. Tournoi et pression selective

`tournament_k` regle la pression selective :

- `k=2` : tournoi doux -> plus de diversite.
- `k=5` ou plus : forte pression -> convergence plus rapide mais risque de converger prematurement.


In [8]:
for k in (2, 3, 5, 7):
    sol = genetic_algorithm(si, population_size=30, generations=40, tournament_k=k, seed=1)
    print(f"k={k} -> cout={sol.total_cost:.2f}")


k=2 -> cout=1448.30


k=3 -> cout=1448.30


k=5 -> cout=1448.30


k=7 -> cout=1448.30


## 8. Reproductibilite


In [9]:
a = genetic_algorithm(si, population_size=20, generations=20, seed=7)
b = genetic_algorithm(si, population_size=20, generations=20, seed=7)
print("identiques ?", a.total_cost == b.total_cost and a.routes == b.routes)


identiques ? True


## 9. En resume

- Le codage giant-tour + Split permet de manipuler des **permutations** (operateurs simples) tout en obtenant des solutions VRP optimales sous l'ordre.
- Sans le schema memetique, le GA peut rester a la traine par rapport au tabou ou au SA.
- Avec memetique, il devient souvent le meilleur candidat sur les instances moyennes, au prix du temps de calcul.
